# Numerical Checks for Wedge Expansion

In [2]:
import numpy as np
from scipy.integrate import quad
from scipy.special import gamma, digamma, factorial
from mpmath import hyper

import matplotlib.pyplot as plt

In [3]:
def h(p):
    if p < 1e-10 or p > 1-1e-10:
        return 0
    return -p*np.log(p)-(1-p)*np.log(1-p)

### Checks for $f_\alpha(x,\epsilon)$ and $g_\beta(x,\epsilon)$.

In [4]:
def g_beta(x, eps, beta):
    return quad(lambda s: (x*s)**beta*np.log(x*s)*(x*s)/(np.sqrt(s**2-1)), 1, np.sqrt(eps**2/x**2 + 1))


def C1(beta):
    if (beta % 2) == 0:
        return np.pi * 2**beta * factorial(beta/2) * factorial(beta/2 + 1) / factorial(beta + 2)
    return np.sqrt(np.pi)/4 * (gamma(-(beta+1)/2)/gamma(-beta/2)) * (digamma(-beta/2) - digamma(-(beta+1)/2))


def C2(beta):
    return np.sqrt(np.pi)/2 * (gamma(-(beta+1)/2)/gamma(-beta/2))

def g_beta_paper(x, eps, beta):
    # Equation (104)
    p1 = C1(beta)*x**(beta+1) + C2(beta)*x**(beta+1)*np.log(x)
    p2 = - 1/(beta+1)**2 * eps**(beta+1) + 1/(beta+1)*eps**(beta+1)*np.log(eps)
    p3 = -1/(2*(beta-1)**2) * x**2 * eps**(beta-1)  + beta/(2*(beta-1)) * x**2 * eps**(beta-1) * np.log(eps)
    p4 = (beta**2 - 6*beta + 6)/(8*(beta - 3)**2) * x**4 * eps**(beta-3) + beta*(beta-2)/(8*(beta-3)) * x**4 * eps**(beta-3) * np.log(eps)
    return p1 + p2 + p3 + p4


def f_alpha_paper(alpha, x, eps):
    return C2(alpha) * x**(alpha+1) + eps**(alpha+1)/(alpha+1) + alpha*eps**(alpha-1)/(2*(alpha - 1))*x**2 + alpha*(alpha-2)*eps**(alpha-3)/(8*(alpha-3)) * x**4


def f_alpha_numeric(alpha, x, eps):
    return quad(lambda s: (x*s)**alpha * (x*s)/(np.sqrt(s**2-1)), 1, np.sqrt(eps**2/x**2 + 1))

In [5]:
for beta in [0.5, 1.5, 2, 2.5, 3.5]:
    x, eps = 0.05, 0.5
    numeric = g_beta(x, eps,beta)
    print(f"Alpha = {beta}: f_alpha (numerical) = {f_alpha_numeric(beta,x,eps)[0]}, f_alpha (paper) = {f_alpha_paper(beta,x,eps)}, difference = {np.abs(f_alpha_numeric(beta,x,eps)[0]-f_alpha_paper(beta,x,eps))}")
    print(f"Beta = {beta}: g_beta (numerical) = {numeric[0]}, g_beta (paper) = {g_beta_paper(x,eps,beta)}, difference = {np.abs(numeric[0]-g_beta_paper(x,eps,beta))}\n")

Alpha = 0.5: f_alpha (numerical) = 0.24370764654531055, f_alpha (paper) = 0.24370765082162094, difference = 4.276310389128568e-09
Beta = 0.5: g_beta (numerical) = -0.3272506913154193, g_beta (paper) = -0.3272506888442251, difference = 2.4711941937205495e-09

Alpha = 1.5: f_alpha (numerical) = 0.07296156298480713, f_alpha (paper) = 0.07296156495073296, difference = 1.965925824909398e-09
Beta = 1.5: g_beta (numerical) = -0.08014047877444715, g_beta (paper) = -0.08014048298074442, difference = 4.206297268827264e-09

Alpha = 2: f_alpha (numerical) = 0.04291666666666598, f_alpha (paper) = 0.042916666666666665, difference = 6.869504964868156e-16
Beta = 2: g_beta (numerical) = -0.04413367723216981, g_beta (paper) = -0.0441336806940203, difference = 3.461850493768903e-09

Alpha = 2.5: f_alpha (numerical) = 0.026005072118598975, f_alpha (paper) = 0.026005070740880563, difference = 1.377718411577078e-09
Beta = 2.5: g_beta (numerical) = -0.02543679572183837, g_beta (paper) = -0.02543679770757321,

In [6]:
def g1(x, eps):
    p1 = -eps**2/4 + eps**2 * np.log(eps)/2
    p2 = x**2/4*np.log(eps)**2 + 1/2*x**2*np.log(eps)
    p3 = x**2/2 * np.log(2/np.sqrt(np.exp(1)*x))*np.log(x) + x**2 * (1/4 + 3/32 * hyper([1,1,1,5/2], [2,2,3], 1))
    p4 = 1/32 * x**4/eps**2 + 1/16 *x**4/eps**2 * np.log(eps)  
    return p1+p2+p3+p4


def g3(x,eps):
    p1 = -1/16 * eps**4 + eps**4/4 * np.log(eps) - 1/8*x**2*eps**2 + 3/4 * x**2 * eps**2 * np.log(eps)
    p2 = x**4 * (1/2*np.log(eps) + 3/16 * np.log(eps)**2 + 5/16 + 5/64*hyper([1,1,1,7/2], [2,2,4], 1))
    p3 = x**4*np.log(x)*(-7/32 + 3/8*np.log(2))
    p4 = -3/16*x**4*np.log(x)**2
    return p1 + p2 + p3 + p4


def f1(x, eps):
    return eps**2/2 + 1/2*np.log(2*np.sqrt(np.exp(1))*eps/x)*x**2 + x**4/(16*eps**2)

def f3(x,eps):
    return eps**4/4 + 3*eps**2/4*x**2 + 3/8*np.log(2*np.exp(3/4)*eps/x)*x**4

### When $\alpha$ and $\beta$ are odd integers:

In [7]:
for x, eps in [(0.01, 0.1), (0.01, 0.25), (0.05, 0.25), (0.05, 0.5)]:
    print(f"(x, eps) = ({x}, {eps})")
    numeric = g_beta(x, eps, 1)
    print(f"Beta = {1}: g_1 (numerical) = {numeric[0]}, g_1 (paper) = {g1(x,eps)}, difference = {np.abs(numeric[0]-g1(x,eps))}")
    numeric = g_beta(x, eps, 3)
    print(f"Beta = {3}: g_3 (numerical) = {numeric[0]}, g_3 (paper) = {g3(x,eps)}, difference = {np.abs(numeric[0]-g3(x,eps))}\n")


(x, eps) = (0.01, 0.1)
Beta = 1: g_1 (numerical) = -0.014531904120272707, g_1 (paper) = -0.0145319044913436, difference = 3.71070887558722e-10
Beta = 3: g_3 (numerical) = -6.570497405360278e-05, g_3 (paper) = -6.57055317541463e-5, difference = 5.57700543474889e-10

(x, eps) = (0.01, 0.25)
Beta = 1: g_1 (numerical) = -0.059504261621685026, g_1 (paper) = -0.0595042616275217, difference = 5.83664366837766e-12
Beta = 3: g_3 (numerical) = -0.001605263938887618, g_3 (paper) = -0.00160526393866614, difference = 2.21482554518815e-13

(x, eps) = (0.05, 0.25)
Beta = 1: g_1 (numerical) = -0.06485668668644261, g_1 (paper) = -0.0648567767535116, difference = 9.00670690112682e-8
Beta = 3: g_3 (numerical) = -0.001790636733020472, g_3 (paper) = -0.00179063247873004, difference = 4.25429042710292e-9

(x, eps) = (0.05, 0.5)
Beta = 1: g_1 (numerical) = -0.15508265040064897, g_1 (paper) = -0.155082653416618, difference = 3.01596855822872e-9
Beta = 3: g_3 (numerical) = -0.015149936314000485, g_3 (paper) = 

### Full check of $f(x,\epsilon) = \sum_{\alpha} a_\alpha f_\alpha(x,\epsilon) + \sum_\beta b_\beta g_\beta(x,\epsilon)$

In [8]:
def f_numeric(x,eps,p):
    return quad(lambda t: h(p(np.sqrt(x**2+t**2))), 0, eps)


def f_paper(x,eps,expo,p):
    p1 = eps * h(p(0))

    power_coeffs = [1, -1/2, 5/24, -1/6, 41/2880]
    log_coeffs = [-expo, expo/2, -expo/6, expo/24, -expo/120]
    exponents = [expo, 2*expo, 3*expo, 4*expo, 5*expo]

    p2 = np.sum([power_coeffs[a] * f_alpha_paper(exponents[a], x, eps) for a in range(len(power_coeffs))])
    p3 = np.sum([log_coeffs[b] * g_beta_paper(x, eps, exponents[b]) for b in range(len(log_coeffs))])

    return p1 + p2 + p3

1. $f(x,\epsilon)$ when $p$ has no integers in the expansion ($e^{-r^{2.37}}$)

In [9]:
for x, eps in [(0.01, 0.1), (0.025, 0.25), (0.025, 0.5), (0.05, 0.5)]:
    numeric = f_numeric(x, eps, lambda r: np.exp(-r**2.37))
    paper = f_paper(x, eps, 2.37, lambda r: np.exp(-r**2.37))
    print(f"x = {x}, eps = {eps}: f (numerical) = {numeric[0]}, f (paper) = {paper}, difference = {np.abs(numeric[0]-paper)}\n")

x = 0.01, eps = 0.1: f (numerical) = 0.000932110989873244, f (paper) = 0.00093211099649591, difference = 6.622665957300988e-12

x = 0.025, eps = 0.25: f (numerical) = 0.014112406161627539, f (paper) = 0.014112402523783402, difference = 3.637844137038826e-09

x = 0.025, eps = 0.5: f (numerical) = 0.09195294138713009, f (paper) = 0.09194588945129377, difference = 7.051935836319134e-06

x = 0.05, eps = 0.5: f (numerical) = 0.0939662646866691, f (paper) = 0.09395897912481227, difference = 7.285561856834599e-06



2. $f(x,\epsilon)$ when $p$ has even integers in the expansion ($e^{-r^2}$)

In [10]:
for x, eps in [(0.01, 0.1), (0.025, 0.25), (0.025, 0.5), (0.05, 0.5)]:
    numeric = f_numeric(x, eps, lambda r: np.exp(-r**2))
    paper = f_paper(x, eps, 2, lambda r: np.exp(-r**2))
    print(f"x = {x}, eps = {eps}: f (numerical) = {numeric[0]}, f (paper) = {paper}, difference = {np.abs(numeric[0]-paper)}, integration error = {numeric[1]}")


x = 0.01, eps = 0.1: f (numerical) = 0.00214843419254782, f (paper) = 0.0021484342373195367, difference = 4.477171671596336e-11, integration error = 2.977515494810576e-11
x = 0.025, eps = 0.25: f (numerical) = 0.023420823165271512, f (paper) = 0.023420777689700294, difference = 4.547557121842294e-08, integration error = 5.705053477924912e-10
x = 0.025, eps = 0.5: f (numerical) = 0.11998022933147971, f (paper) = 0.11995737557339883, difference = 2.2853758080879083e-05, integration error = 5.705064198172494e-10
x = 0.05, eps = 0.5: f (numerical) = 0.1227433794387317, f (paper) = 0.12271964171985611, difference = 2.373771887559384e-05, integration error = 5.91821930396261e-09


3. $f(x,\epsilon)$ when $p$ has odd integers in the expansion ($e^{-r}$)

In [11]:
def f_paper_odd(x, eps, expo, p):
    p1 = eps * h(p(0))

    power_coeffs = [1, -1/2, 5/24, -1/6, 41/2880]
    log_coeffs = [-expo, expo/2, -expo/6, expo/24, -expo/120]
    exponents = [expo, 2*expo, 3*expo, 4*expo, 5*expo]

    p2, p3 = 0, 0
    for a in range(len(power_coeffs)):
        if exponents[a]==1:
            p2 += power_coeffs[a] * f1(x, eps)
            p3 += log_coeffs[a] * g1(x, eps)
        elif exponents[a]==3:
            p2 += power_coeffs[a] * f3(x, eps)
            p3 += log_coeffs[a] * g3(x, eps)
        elif (exponents[a]+1) % 2 ==0:
            continue
        else:
            p2 += power_coeffs[a] * f_alpha_paper(exponents[a], x, eps)
            p3 += log_coeffs[a] * g_beta_paper(x, eps, exponents[a])

    return p1 + p2 + p3

In [12]:
for x, eps in [(0.01, 0.1), (0.025, 0.25), (0.025, 0.5), (0.05, 0.5)]:
    numeric = f_numeric(x, eps, lambda r: np.exp(-r))
    paper = f_paper_odd(x, eps, 1, lambda r: np.exp(-r))
    print(f"x = {x}, eps = {eps}: f (numerical) = {numeric[0]}, f (paper) = {paper}, difference = {np.abs(numeric[0]-paper)}\n")

x = 0.01, eps = 0.1: f (numerical) = 0.019098244714473932, f (paper) = 0.0190980239153026, difference = 2.20799171307684e-7

x = 0.025, eps = 0.25: f (numerical) = 0.08669083955463806, f (paper) = 0.0866687034554059, difference = 2.21360992321284e-5

x = 0.025, eps = 0.5: f (numerical) = 0.24032705035789567, f (paper) = 0.239618312756282, difference = 0.000708737601613468

x = 0.05, eps = 0.5: f (numerical) = 0.24621875522935602, f (paper) = 0.245492038250853, difference = 0.000726716978502545



### Full check of $F(x)$.

In [13]:
def rho(r,p):
    return h(p(r))


def rhoprime(r, p):
    return -2*r *np.exp(-r**2) * np.log((1-p(r)/p(r)))


def F_numeric(x, rho):
    return quad(lambda t: rho(np.sqrt(x**2+t**2)), 0, np.inf)


def rhok(k, rho):
    return quad(lambda r: r**k * rho(r), 0, np.inf)


def rhoprimek(k, rho):
    return quad(lambda r: r**k * rhoprime(r), 0, np.inf)



def F_analytic(x, expo, p):
    p1 = rhok(0, rho=lambda r: h(p(r)))[0]
    power_coeffs = [1, -1/2, 5/24, -1/6, 41/2880]
    log_coeffs = [-expo, expo/2, -expo/6, expo/24, -expo/120]
    exponents = [expo, 2*expo, 3*expo, 4*expo, 5*expo]

    p2 = np.sum([power_coeffs[a] * C2(exponents[a]) * x**(exponents[a]+1) for a in range(len(power_coeffs))])
    p3 = np.sum([log_coeffs[b]* (C1(exponents[b]) + C2(exponents[b])*np.log(x)) * x**(exponents[b]+1) for b in range(len(log_coeffs))])

    return p1 + p2 + p3


def p_tilde_1(eps, expo, p):
    power_coeffs = [1, -1/2, 5/24, -1/6, 41/2880]
    log_coeffs = [-expo, expo/2, -expo/6, expo/24, -expo/120]
    exponents = [expo, 2*expo, 3*expo, 4*expo, 5*expo]
    p1 = quad(lambda r: 1/r * h(p(r)), eps, np.inf)[0]
    p2 = np.sum([power_coeffs[a] * exponents[a]/(exponents[a]-1) * eps**(exponents[a]-1) for a in range(1, len(power_coeffs))])
    p3 = np.sum([log_coeffs[b] * eps**(exponents[b]-1) * (exponents[b] * np.log(eps) / (exponents[b]-1) - 1/(exponents[b]-1)**2) for b in range(1, len(log_coeffs))])
    p4 = power_coeffs[0] * np.log(2*np.abs(power_coeffs[0]) * np.exp(1/2) * eps)
    p5 = log_coeffs[0] * (1/2 + np.log(eps) + 1/2 * np.log(eps)**2 + 3/16 * hyper([1,1,1,5/2],[2,2,3],1))
    return p1 + p2 + p3 +p4 + p5


def p_tilde_3(eps, expo, p):
    power_coeffs = [1, -1/2, 5/24, -1/6, 41/2880]
    log_coeffs = [-expo, expo/2, -expo/6, expo/24, -expo/120]
    exponents = [expo, 2*expo, 3*expo, 4*expo, 5*expo]
    p1 = eps**(-2)*(-expo * eps**(expo-1) * p(eps) * np.log((1-p(eps))/p(eps))) + 2*eps**(-3)*h(p(eps)) + 6 * quad(lambda r: 1/r**4 * h(p(r)), eps, np.inf)[0]
    p2 = - quad(lambda r: r**(-3) * (-expo * eps**(expo-1) * p(eps) * np.log((1-p(eps))/p(eps))), eps,np.inf)[0]
    p3, p4 = 0, 0
    for a in range(len(power_coeffs)):
        alpha=exponents[a]
        if alpha == 3:
            continue
        p3 += power_coeffs[a] * eps**(alpha-3) * alpha*(alpha-2)/(alpha-3)
        p4 += log_coeffs[a] * eps**(alpha-3) * ( alpha*(alpha-2)/(alpha-3)*np.log(eps) + (alpha**2-6*alpha+6)/(8*(alpha-3)**2))
    p5 = 3*power_coeffs[1] * np.log(2*np.abs(power_coeffs[1]) * np.exp(3/4)*eps)
    p6 = log_coeffs[1] * (4*np.log(eps) + 3/2*np.log(eps)**2)
    return p1 + p2 + p3 + p4 + p5 + p6



def F_analytic_odd(x, expo, p):
    p1 = rhok(0, rho=lambda r: h(p(r)))[0]
    power_coeffs = [1, -1/2, 5/24, -1/6, 41/2880]
    log_coeffs = [-expo, expo/2, -expo/6, expo/24, -expo/120]
    exponents = [expo, 2*expo, 3*expo, 4*expo, 5*expo]

    p2, p3 = 0,0
    for a in range(len(power_coeffs)):
        if exponents[a] == 1:
            p2 += x**2/2 * (-power_coeffs[a] * np.log(np.abs(power_coeffs[a])*x) + log_coeffs[a] * np.log(x) * np.log(2/np.sqrt(x)/np.exp(1/2)))
        elif exponents[a] == 3:
            p2 += x**4/8 * (log_coeffs[a] * ((3*np.log(2)-7/4) * np.log(x) + 5/2 + 5/8 * hyper([1,1,1,7/2], [2,2,4], 1) - 3/2*np.log(x)**2) - 3*power_coeffs[a] * np.log(np.abs(power_coeffs[a])*x))
        elif (exponents[a] + 1) % 2 ==0:
            continue
        else:
            p2 += power_coeffs[a] * C2(exponents[a]) * x **(exponents[a]+1)
            p3 += log_coeffs[a] * x**(exponents[a]+1) * (C1(exponents[a]) + C2(exponents[a])*np.log(x))
    
    return p1 + p2 + p3

1. $F(x)$ when $p$ has no integers in the expansion ($e^{-r^{2.37}}$)

In [14]:
xs = [0.01, 0.025, 0.05, 0.1]

for x in xs:
    numeric = F_numeric(x, rho = lambda r: h(np.exp(-r**2.37)))
    paper = F_analytic(x, 2.37, p=lambda r:np.exp(-r**2.37))
    print(f"x = {x}: F (numerical) = {numeric[0]}, F (paper) = {paper}, difference = {np.abs(numeric[0]-paper)}\n")

x = 0.01: F (numerical) = 0.7114048358659391, F (paper) = 0.7113047547054991, difference = 0.00010008116043991322

x = 0.025: F (numerical) = 0.7119336421501896, F (paper) = 0.7113076561429514, difference = 0.0006259860072381773

x = 0.05: F (numerical) = 0.7138201735649965, F (paper) = 0.711309443026795, difference = 0.0025107305382015

x = 0.1: F (numerical) = 0.7212155352541401, F (paper) = 0.7110673201450468, difference = 0.01014821510909325



2. $F(x)$ when $p$ has even integers in the expansion ($e^{-r^{2}}$)

In [15]:
for x in xs:
    numeric = F_numeric(x, rho = lambda r: h(np.exp(-r**2)))
    paper = F_analytic(x, 2, p=lambda r:np.exp(-r**2))
    print(f"x = {x}: F (numerical) = {numeric[0]}, F (paper) = {paper}, difference = {np.abs(numeric[0]-paper)}\n")


x = 0.01: F (numerical) = 0.822889984244565, F (paper) = 0.8227407437816797, difference = 0.0001492404628853361

x = 0.025: F (numerical) = 0.8236428417221351, F (paper) = 0.822710121349304, difference = 0.0009327203728310973

x = 0.05: F (numerical) = 0.8262117172895527, F (paper) = 0.8224813003177017, difference = 0.0037304169718509472

x = 0.1: F (numerical) = 0.8355710339218183, F (paper) = 0.8206567966881754, difference = 0.014914237233642935



3. $F(x)$ when $p$ has odd integers in the expansion ($e^{-r}$)

In [16]:
for x in xs:
    numeric = F_numeric(x, rho = lambda r: h(np.exp(-r)))
    paper = F_analytic_odd(x, 1, p=lambda r:np.exp(-r))
    print(f"x = {x}: F (numerical) = {numeric[0]}, F (paper) = {paper}, difference = {np.abs(numeric[0]-paper)}\n")

x = 0.01: F (numerical) = 1.6456843452593946, F (paper) = 1.6457395210934, difference = 5.51758340086028e-5

x = 0.025: F (numerical) = 1.648099308614067, F (paper) = 1.64844420830214, difference = 0.00034489968807061

x = 0.05: F (numerical) = 1.6536998106607026, F (paper) = 1.65508013492837, difference = 0.00138032426767043

x = 0.1: F (numerical) = 1.6669577664783415, F (paper) = 1.67249065302941, difference = 0.00553288655107087

